# Rebuild Task 1: Tiny semantic search, by hand

**Chapter:** Hands-On LLMs, Ch 2 (tokens and token embeddings)
**Priority:** MUST KNOW (do every rebuild task for Ch 1-2 and Ch 8)
**DE skill this builds:** the core mechanic under every RAG retriever.

DE mapping for this task:
- chunk = row grain
- embedding = derived column
- embedding model = the UDF that makes that column
- search = a scan with a distance function

**Rules**
- Close the book notebook. Write this from your head.
- Plain Python + numpy only.
- `sentence-transformers` is allowed to produce embeddings (it is the UDF, not a framework).
- NO sklearn `cosine_similarity`, NO FAISS, NO LangChain / LlamaIndex.
- Hints before answers. Paste your code back to me when done and I will review it.

## Warm-up (answer before coding)

**What do you think an embedding is actually storing?** Honest answer, no googling. I will poke at it.

*Your answer (double click to edit):*

The embedding is storing a vector representation of the tokenized statement.

## Setup

In [2]:
%pip install sentence-transformers numpy pytest


Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Imports. Add what you need.
import numpy as np
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer


## The corpus

A **corpus** is just the pile of texts you search over. Think of it as your source table: one row per text.

Seeded on purpose:
- Part A material: a SQL snippet (`d10`), a typo (`d11`), emoji (`d12`), odd unicode (`d13`)
- Part D material: topic clusters (Airflow, Trino, Delta, Spark, data quality) so you can write paraphrase queries
- Two off-topic distractors (`d23`, `d24`)

Keep `CORPUS` as the single source of truth. Derive `ids` and `texts` from it, do not hand-maintain three lists.

In [ ]:
CORPUS: list[dict[str, str]] = [
    {"id": "d01", "text": "Airflow retries a failed task up to three times with a five minute delay between attempts."},
    {"id": "d02", "text": "Use sensors sparingly in Airflow because they hold a worker slot while they wait."},
    {"id": "d03", "text": "A DAG with catchup enabled will backfill every missed schedule interval since its start date."},
    {"id": "d04", "text": "Trino joins are fastest when the smaller table is broadcast to every worker node."},
    {"id": "d05", "text": "Partition pruning in Trino skips files whose partition values cannot match the WHERE clause."},
    {"id": "d06", "text": "Running EXPLAIN ANALYZE in Trino shows per-stage wall time, row counts, and data skew."},
    {"id": "d07", "text": "Delta Lake VACUUM permanently removes data files no longer referenced by the transaction log."},
    {"id": "d08", "text": "Delta time travel lets you query a table as of a previous version or timestamp."},
    {"id": "d09", "text": "Z-ordering co-locates related rows in the same files so selective queries read less data."},
    {"id": "d10",
        "text": "SELECT u.user_id, COUNT(*) AS n FROM events e JOIN users u ON e.user_id = u.user_id WHERE e.ds >= DATE '2026-01-01' GROUP BY 1 HAVING COUNT(*) > 10;"},
    {"id": "d11", "text": "Our pipline failed becuase the upstream schmea changed and nobody was notifed."},
    {"id": "d12", "text": "Deploy went out Friday afternoon \U0001F680\U0001F525 and the on-call pager was NOT happy \U0001F62C"},
    {"id": "d13", "text": "Customer name arrived as \u00dcber-M\u00fcller \u2014 mojibake showed up as \u00c3\u0153ber after the CSV export."},
    {"id": "d14", "text": "PySpark shuffles data across the cluster during groupBy, join, and repartition operations."},
    {"id": "d15", "text": "Broadcast variables ship a read-only copy of a lookup table to each executor once."},
    {"id": "d16", "text": "Data skew happens when a few keys hold most of the rows, leaving one task running long after the rest finish."},
    {"id": "d17", "text": "Cache a DataFrame only if it is reused several times, otherwise you just burn executor memory."},
    {"id": "d18", "text": "Idempotent pipelines can be rerun for the same date without creating duplicate rows."},
    {"id": "d19", "text": "A row count check against the source table catches silently dropped records after a load."},
    {"id": "d20", "text": "Freshness alerts fire when the newest partition is older than the agreed SLA."},
    {"id": "d21", "text": "Schema evolution lets a table accept new columns without rewriting historical data."},
    {"id": "d22", "text": "Slowly changing dimensions track history by adding a new row with effective start and end dates."},
    {"id": "d23", "text": "Let the sourdough starter double in size before mixing it into the dough."},
    {"id": "d24", "text": "A full-body strength session usually pairs one squat or hinge with one push and one pull."},
]


## Part A: Poke the tokenizer (Ch 2)

**Production angle:** tokens, not characters, are what hit the model's input limit. This is how you will eventually size chunks.

**Steps**
1. Pick your embedding model (any `sentence-transformers` model, for example `all-MiniLM-L6-v2`). Write down which one.
2. Load the tokenizer that belongs to that model. Not a random one, the matching one.
3. For 5 texts, print the tokens and the token count. Must include: one SQL snippet, one typo, one emoji or odd unicode.
4. Also print the model's max sequence length. Compare it to your longest text in tokens.

**Hint (peek only if stuck):** the `SentenceTransformer` object exposes its tokenizer, and has a max sequence length attribute.

In [ ]:
# TODO: load model + tokenizer, print tokens and counts for 5+ texts
tokenizer = AutoTokenizer.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

sublist = {
    CORPUS[9]["id"]: CORPUS[9]["text"],
    CORPUS[10]["id"]: CORPUS[10]["text"],
    CORPUS[11]["id"]: CORPUS[11]["text"],
    CORPUS[12]["id"]: CORPUS[12]["text"],
    CORPUS[13]["id"]: CORPUS[13]["text"],
}

token_dict = {}
for doc_id, text in sublist.items():
    tokens = tokenizer.tokenize(text)
    token_dict[doc_id] = {
        "tokens": tokens,
        "count": len(tokens)
    }


In [ ]:
for doc_id, info in token_dict.items():
    print(f"Document ID: {doc_id}")
    print(f"Tokens: {info['tokens']}")
    print(f"Token Count: {info['count']}\n")


Document ID: d10
Tokens: ['select', 'u', '.', 'user', '_', 'id', ',', 'count', '(', '*', ')', 'as', 'n', 'from', 'events', 'e', 'join', 'users', 'u', 'on', 'e', '.', 'user', '_', 'id', '=', 'u', '.', 'user', '_', 'id', 'where', 'e', '.', 'ds', '>', '=', 'date', "'", '202', '##6', '-', '01', '-', '01', "'", 'group', 'by', '1', 'having', 'count', '(', '*', ')', '>', '10', ';']
Token Count: 57

Document ID: d11
Tokens: ['our', 'pip', '##line', 'failed', 'be', '##cu', '##ase', 'the', 'upstream', 'sc', '##hm', '##ea', 'changed', 'and', 'nobody', 'was', 'not', '##ife', '##d', '.']
Token Count: 20

Document ID: d12
Tokens: ['deploy', 'went', 'out', 'friday', 'afternoon', '[UNK]', 'and', 'the', 'on', '-', 'call', 'page', '##r', 'was', 'not', 'happy', '[UNK]']
Token Count: 17

Document ID: d13
Tokens: ['customer', 'name', 'arrived', 'as', 'uber', '-', 'muller', '—', 'mo', '##ji', '##ba', '##ke', 'showed', 'up', 'as', 'a', '##œ', '##ber', 'after', 'the', 'cs', '##v', 'export', '.']
Token Count: 

In [ ]:
vector_dict = {}
for doc_id, text in sublist.items():
    vector = model.encode(text)
    vector_dict[doc_id] = vector


In [ ]:
tokenized_texts = {}
for doc_id, text in sublist.items():
    tokens = model.tokenizer(text, return_tensors="pt")
    tokenized_texts[doc_id] = tokens


In [ ]:
for doc_id, info in tokenized_texts.items():
    print(f"Document ID: {doc_id}")
    print(f"Tokenized Input IDs: {info['input_ids']}")
    print(f"Number of Tokens: {info['input_ids'].size(1)}\n")


Document ID: d10
Tokenized Input IDs: tensor([[  101,  7276,  1057,  1012,  5310,  1035,  8909,  1010,  4175,  1006,
          1008,  1007,  2004,  1050,  2013,  2824,  1041,  3693,  5198,  1057,
          2006,  1041,  1012,  5310,  1035,  8909,  1027,  1057,  1012,  5310,
          1035,  8909,  2073,  1041,  1012, 16233,  1028,  1027,  3058,  1005,
         16798,  2575,  1011,  5890,  1011,  5890,  1005,  2177,  2011,  1015,
          2383,  4175,  1006,  1008,  1007,  1028,  2184,  1025,   102]])
Number of Tokens: 59

Document ID: d11
Tokenized Input IDs: tensor([[  101,  2256, 28315,  4179,  3478,  2022, 10841, 11022,  1996, 13909,
          8040, 14227,  5243,  2904,  1998,  6343,  2001,  2025, 29323,  2094,
          1012,   102]])
Number of Tokens: 22

Document ID: d12
Tokenized Input IDs: tensor([[  101, 21296,  2253,  2041,  5958,  5027,   100,  1998,  1996,  2006,
          1011,  2655,  3931,  2099,  2001,  2025,  3407,   100,   102]])
Number of Tokens: 19

Document ID: d1

In [ ]:
model.tokenizer.max_len_single_sentence
# 254, longest text in sublist is 148.


254

In [ ]:
for text in sublist.values():
    print(len(text))


148
78
73
88
90


### Part A notes: what surprised you? (3 bullets)

- Realizing the SentenceTransformer has it's own Tokenizer
- TODO
- TODO

## Part B: Embed it

**Steps**
1. Embed every text in the corpus.
2. L2-normalize the vectors **once, at index time**.
3. Store them as a single `(N, d)` numpy matrix, with a parallel list of ids (same order).
4. Print the shape and the dtype.

**Hint:** `np.linalg.norm(..., axis=1, keepdims=True)`.

**Think about:** if you changed the embedding model tomorrow, what would have to change? (Hello, `embedding_version`. We will formalize this in the ingestion task.)

In [ ]:
# TODO: build ids, texts, and the normalized (N, d) matrix
full_corpus_embeddings = model.encode(
    [doc["text"] for doc in CORPUS], normalize_embeddings=True)


In [ ]:
print(full_corpus_embeddings.shape)
print(full_corpus_embeddings.dtype)


(24, 384)
float32


## Part C: Search

**Steps**
1. Implement `search(query, k)` using only numpy for the scoring and ranking.
2. Embed the query, normalize it, then score all docs with one matrix-vector dot product. No Python loop over docs.
3. Pick top-k with `argpartition` or `argsort`. Be ready to say why you picked one.
4. Return `list[tuple[str, float]]` of `(doc_id, score)`, best first.

**Must have:** type hints, a docstring, and a clear return contract (sorted? what if `k > N`? what if `k <= 0`?).

**Be able to explain:** why does normalizing turn cosine similarity into a plain dot product?

In [ ]:
def search(query: str, k: int) -> list[tuple[str, float]]:
    """TODO: write the docstring and the return contract."""
    raise NotImplementedError


### Part C tests (3 minimum)

Ideas (pick or swap):
1. A corpus text queried against itself returns itself at rank 1 with a score near 1.0.
2. Output has the right length and is sorted descending.
3. `k > N` behaves sanely (decide what "sane" means, then pin it with a test).

Write them as plain `test_*` functions in the cell below, then run them in the cell after that.

In [ ]:
# TODO: test_* functions


In [ ]:
# Run your tests. Adjust if you move the tests to a separate file.
import pytest

# pytest.main(["-q", "-p", "no:cacheprovider", "--no-header", "-x", "<path_to_test_file>"])


## Part D: Break it

Run exactly 5 queries:
- 2 that **share keywords** with corpus texts
- 2 that are **paraphrases with zero shared keywords** (no word overlap with the doc you are targeting)
- 1 that is **total nonsense**

For each: print top 3 with scores.

**Hint:** if your nonsense query still returns confident-looking scores, that is a real finding. Keep it, we use it in the recall@k eval set later.

In [ ]:
# TODO: run the 5 queries, print top 3 each


### Part D write-up

**Where results looked good:**
- TODO

**Where results looked bad:**
- TODO

**What the scores did on the nonsense query:**
- TODO

**One honest "this retrieval is wrong and here is my guess why":**
- TODO

## What good looks like (self-grade)

- [ ] Tokenizer output printed for 5+ texts, with your own observations
- [ ] Vectors normalized once at index time, not on every query
- [ ] Cosine similarity done as a matrix-vector dot product, no Python loop over docs
- [ ] `search` has type hints, a docstring, and a clear return contract
- [ ] Top-k uses `argpartition` or `argsort` on scores, and you can say why you picked one
- [ ] Tests pass, including the `k > N` edge case
- [ ] You can explain why normalizing turns cosine into a dot product
- [ ] Part D notes include one honest "this retrieval is wrong and here is my guess why"

When you are done, paste your code in chat and I will review it.